# Neural Networks & Generative AI Assignment

This notebook completes the required parts of the Module 10 assignment.

Dataset: Fashion-MNIST  
Framework: TensorFlow / Keras  
Model: Simple CNN  
Generative AI technique used in this assignment: data augmentation with Keras preprocessing layers

Note: Run this notebook in Google Colab or another environment with TensorFlow installed.

## Part A: Build a Simple Neural Network

Fashion-MNIST contains 60,000 training images and 10,000 test images. Each image is a 28x28 grayscale clothing item labeled into one of 10 classes.

In [ ]:
# If you are running in Colab, TensorFlow is usually already installed.
# If needed, uncomment the next line:
# !pip install -q tensorflow

import random
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

from tensorflow import keras
from tensorflow.keras import layers

print("TensorFlow version:", tf.__version__)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [ ]:
# Load the dataset
(x_train, y_train), (x_test, y_test) = keras.datasets.fashion_mnist.load_data()

class_names = [
    "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"
]

print("Training shape:", x_train.shape)
print("Test shape:", x_test.shape)

In [ ]:
# Normalize and reshape
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

x_train = np.expand_dims(x_train, -1)  # (60000, 28, 28, 1)
x_test = np.expand_dims(x_test, -1)    # (10000, 28, 28, 1)

print("Reshaped training shape:", x_train.shape)
print("Reshaped test shape:", x_test.shape)

In [ ]:
# Visualize a few training samples
plt.figure(figsize=(10, 5))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(x_train[i].squeeze(), cmap="gray")
    plt.title(class_names[y_train[i]])
    plt.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
def build_baseline_model():
    model = keras.Sequential([
        layers.Input(shape=(28, 28, 1)),
        layers.Conv2D(32, kernel_size=3, activation="relu"),
        layers.MaxPooling2D(pool_size=2),
        layers.Conv2D(64, kernel_size=3, activation="relu"),
        layers.MaxPooling2D(pool_size=2),
        layers.Flatten(),
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.3),
        layers.Dense(10, activation="softmax")
    ])

    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

baseline_model = build_baseline_model()
baseline_model.summary()

In [ ]:
# Train the baseline model
baseline_history = baseline_model.fit(
    x_train,
    y_train,
    epochs=5,
    batch_size=128,
    validation_split=0.1,
    verbose=1
)

baseline_test_loss, baseline_test_acc = baseline_model.evaluate(x_test, y_test, verbose=0)
print(f"Baseline test accuracy: {baseline_test_acc:.4f}")

## Part B: Experiment with Data Augmentation

Instead of the older `ImageDataGenerator` workflow, this notebook uses current Keras image augmentation layers directly in the model pipeline.

In [ ]:
# Data augmentation pipeline
data_augmentation = keras.Sequential([
    layers.RandomRotation(0.08),
    layers.RandomZoom(0.10),
    layers.RandomTranslation(height_factor=0.08, width_factor=0.08),
], name="data_augmentation")

# Preview augmented images
sample_batch = x_train[:9]

plt.figure(figsize=(8, 8))
for i in range(9):
    augmented = data_augmentation(tf.expand_dims(sample_batch[i], 0), training=True)
    plt.subplot(3, 3, i + 1)
    plt.imshow(augmented[0].numpy().squeeze(), cmap="gray")
    plt.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
def build_augmented_model():
    model = keras.Sequential([
        layers.Input(shape=(28, 28, 1)),
        data_augmentation,
        layers.Conv2D(32, kernel_size=3, activation="relu"),
        layers.MaxPooling2D(pool_size=2),
        layers.Conv2D(64, kernel_size=3, activation="relu"),
        layers.MaxPooling2D(pool_size=2),
        layers.Flatten(),
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.3),
        layers.Dense(10, activation="softmax")
    ])

    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

augmented_model = build_augmented_model()
augmented_model.summary()

In [ ]:
# Train the augmented model
augmented_history = augmented_model.fit(
    x_train,
    y_train,
    epochs=5,
    batch_size=128,
    validation_split=0.1,
    verbose=1
)

augmented_test_loss, augmented_test_acc = augmented_model.evaluate(x_test, y_test, verbose=0)
print(f"Augmented test accuracy: {augmented_test_acc:.4f}")

In [ ]:
# Compare the two results
improvement = augmented_test_acc - baseline_test_acc
print(f"Accuracy difference (augmented - baseline): {improvement:.4f}")

plt.figure(figsize=(6, 4))
plt.bar(["Baseline", "Augmented"], [baseline_test_acc, augmented_test_acc])
plt.ylabel("Test Accuracy")
plt.title("Baseline vs. Augmented Model")
plt.ylim(0.0, 1.0)
plt.show()

In [ ]:
# Plot training histories
plt.figure(figsize=(7, 4))
plt.plot(baseline_history.history["val_accuracy"], label="Baseline val_accuracy")
plt.plot(augmented_history.history["val_accuracy"], label="Augmented val_accuracy")
plt.xlabel("Epoch")
plt.ylabel("Validation Accuracy")
plt.title("Validation Accuracy During Training")
plt.legend()
plt.show()

## Part D: Reflection

Use the results from your run to answer the reflection prompts below.

In [ ]:
print("Reflection summary")
print("-" * 60)

if augmented_test_acc > baseline_test_acc:
    print(f"Data augmentation improved test accuracy by {improvement:.4f}.")
elif augmented_test_acc < baseline_test_acc:
    print(f"Data augmentation reduced test accuracy by {abs(improvement):.4f}.")
else:
    print("Data augmentation produced the same test accuracy as the baseline model.")

print()
print("What I learned:")
print("Data augmentation can improve generalization by exposing the model to more variation during training.")
print()
print("Challenges:")
print("Training takes longer with augmentation, and performance depends on choosing realistic transformations.")
print()
print("Real-world use:")
print("These techniques are useful when real labeled data is limited or when a model must handle variation in images captured under different conditions.")

### Written reflection paragraph you can submit

In this assignment, I built a simple convolutional neural network on the Fashion-MNIST dataset and then retrained it using data augmentation. The baseline model learned the dataset quickly and produced a solid test accuracy. After adding augmentation, the model saw more varied versions of the training images, which helped test how well it could generalize instead of memorizing the original samples. The main challenge was selecting realistic augmentations. If the transformations are too aggressive, they can make the images less representative of the real dataset and hurt performance. This exercise showed that augmentation can act like a practical generative AI technique because it expands the training data with new examples. In real-world applications, this is useful when labeled data is limited or when a model must perform reliably under different viewing conditions, lighting, or object positions.

## Part E: GitHub Submission

Create a public repository named `neural-networks-generative-ai-assignment` and upload:

1. This notebook
2. The README.md file
3. Any screenshots you want to include

Then submit the GitHub repository link in Canvas.